# Практична робота №3. Очищення та перетворення даних
**Виконав:** Julian Pasovystyy

## Крок 1. Підготовка середовища та завантаження даних

In [ ]:
import pandas as pd
import numpy as np

# У Google Colab монтуємо Диск, у Jupyter працюємо з локальною папкою
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/'
except ImportError:
    BASE = ''

df = pd.read_csv(BASE + 'index_1.csv')
df.head()

## Крок 2. Виявлення та обробка пропущених значень

In [ ]:
data_messy = {
    'name': ['Anna', None, 'Bohdan', 'Olena', None],
    'age': [20, 21, None, 22, 24],
    'score': [85.5, 90.0, 78.3, None, 92.1]
}
df_messy = pd.DataFrame(data_messy)
df_messy

In [ ]:
print(df_messy.isnull().sum())

In [ ]:
# Видалення рядків з пропусками у 'name'
df_dropped = df_messy.dropna(subset=['name'])
print(df_dropped)

# Заповнення пропусків у 'age' середнім значенням
mean_age = df_messy['age'].mean()
df_filled = df_messy.copy()
df_filled['age'] = df_filled['age'].fillna(mean_age)
print(df_filled)

## Крок 3. Виявлення та видалення дублікатів

In [ ]:
data_dup = {
    'id': [1, 2, 3, 2, 3],
    'value': ['A', 'B', 'C', 'B', 'C']
}
df_dup = pd.DataFrame(data_dup)
df_dup

In [ ]:
print(df_dup.duplicated())
print("Кількість дублікатів:", df_dup.duplicated().sum())

In [ ]:
df_unique = df_dup.drop_duplicates()
print(df_unique)

## Крок 4. Виявлення та обробка викидів (метод IQR)

In [ ]:
Q1 = df['money'].quantile(0.25)
Q3 = df['money'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"Q1 = {Q1}, Q3 = {Q3}, IQR = {IQR}")
print(f"Нижня межа: {lower}, Верхня межа: {upper}")

In [ ]:
outliers = df[(df['money'] < lower) | (df['money'] > upper)]
print(f"Знайдено викидів: {len(outliers)}")

## Крок 5. Перетворення типів даних

In [ ]:
print(df.dtypes, "\n")

df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')

print(df.dtypes)

## Крок 6. Обробка рядкових даних

In [ ]:
df_text = pd.DataFrame({
    'city': [' Kyiv ', 'LVIV', 'Odesa ', 'KHARKIV', ' kyiv '],
    'phone': ['(044) 123-45-67', '(032) 234-56-78', '(048) 345-67-89', '(057) 456-78-90', '(044) 123-45-67']
})
df_text

In [ ]:
df_text['city'] = df_text['city'].str.strip().str.lower()
df_text['phone'] = df_text['phone'].str.replace(r'\(|\)|-', '', regex=True)
print(df_text)

## Крок 7. Створення нових ознак (feature engineering)

In [ ]:
df['date_only'] = df['datetime'].dt.date
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek
df['month'] = df['datetime'].dt.month
print(df[['datetime', 'date_only', 'hour', 'day_of_week', 'month']].head())

## Крок 8. One-hot encoding

In [ ]:
df_encoded = pd.get_dummies(df, columns=['coffee_name'])
print(df_encoded.head())

## Крок 9. Нормалізація та стандартизація

In [ ]:
df['money_norm'] = (df['money'] - df['money'].min()) / (df['money'].max() - df['money'].min())
df['money_std'] = (df['money'] - df['money'].mean()) / df['money'].std()

In [ ]:
print(df[['money', 'money_norm', 'money_std']].head())
print("Мінімум money_norm:", df['money_norm'].min())
print("Максимум money_norm:", df['money_norm'].max())
print("Середнє money_std:", df['money_std'].mean())
print("Стандартне відхилення money_std:", df['money_std'].std())

## Крок 10. Збереження очищених даних

In [ ]:
final_df = df[['datetime', 'cash_type', 'money', 'coffee_name',
               'hour', 'day_of_week', 'month', 'money_norm', 'money_std']]
final_df.to_csv(BASE + 'cleaned_sales.csv', index=False)
print("Файл збережено!")

In [ ]:
final_df.head()

# Індивідуальні завдання

### Завдання 1. DataFrame з пропусками, дублікатами та викидами

In [ ]:
df1 = pd.DataFrame({
    'id':     [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    'name':   ['Anna', 'Bohdan', 'Olena', 'Petro', 'Iryna', 'Taras', 'Maria', 'Oleh', 'Sofia', 'Andrii', 'Nazar', 'Yulia'],
    'city':   ['Kyiv', 'Lviv', 'Odesa', 'Kyiv', None, 'Lviv', 'Kyiv', 'Kharkiv', 'Lviv', 'Kyiv', 'Odesa', 'Lviv'],
    'age':    [20, 21, None, 22, 24, 23, 22, None, 21, 25, 22, 23],
    'salary': [30000, 32000, 31000, 29000, None, 33000, 30500, 31500, 250000, 32500, 28500, 29500],
    'score':  [85.5, 90.0, 78.3, None, 92.1, 88.0, 81.5, 79.0, 87.5, 91.0, 84.0, 5.0]
})
# додаємо повні дублікати (рядки 0 і 3)
df1 = pd.concat([df1, df1.iloc[[0, 3]]], ignore_index=True)
df1

In [ ]:
# 1) Пропуски
print(df1.isnull().sum())

# 2) Заповнення: медіана для числових, мода для категоріальних
for col in ['age', 'salary', 'score']:
    df1[col] = df1[col].fillna(df1[col].median())
df1['city'] = df1['city'].fillna(df1['city'].mode()[0])
print(df1.isnull().sum())

In [ ]:
# 3) Повні дублікати
print("Дублікатів:", df1.duplicated().sum())
df1 = df1.drop_duplicates()
print("Рядків після видалення:", len(df1))

In [ ]:
# 4) Викиди за IQR (по числових колонках, крім id)
mask = pd.Series(True, index=df1.index)
for col in ['age', 'salary', 'score']:
    q1, q3 = df1[col].quantile(0.25), df1[col].quantile(0.75)
    iqr = q3 - q1
    low, up = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((df1[col] < low) | (df1[col] > up)).sum()
    print(f"{col}: межі [{low:.2f}; {up:.2f}], викидів: {n_out}")
    mask &= df1[col].between(low, up)

df1_clean = df1[mask]
print("Рядків після фільтрації викидів:", len(df1_clean))
df1_clean

### Завдання 2. Датасет продажів: 5 пропусків, 3 дублікати, очищення

In [ ]:
df2 = pd.read_csv(BASE + 'index_1.csv')
print("Початковий розмір:", df2.shape)

# 5 штучних пропусків у money
rng = np.random.default_rng(42)
nan_idx = rng.choice(df2.index, size=5, replace=False)
df2.loc[nan_idx, 'money'] = np.nan

# 3 штучних дублікати
dup_rows = df2.drop(nan_idx).sample(3, random_state=42)
df2 = pd.concat([df2, dup_rows], ignore_index=True)

print("Після додавання:", df2.shape)
print("Пропусків у money:", df2['money'].isnull().sum())
print("Дублікатів:", df2.duplicated().sum())

In [ ]:
# Очищення
df2['money'] = df2['money'].fillna(df2['money'].mean())   # пропуски -> середнє
df2 = df2.drop_duplicates()                               # видаляємо дублікати
df2['datetime'] = pd.to_datetime(df2['datetime'], errors='coerce')

print("Розмір після очищення:", df2.shape)
print("Пропусків:", df2['money'].isnull().sum(), "| Дублікатів:", df2.duplicated().sum())
print(df2.dtypes)
df2.to_csv(BASE + 'sales_cleaned_task2.csv', index=False)

### Завдання 3. Нові ознаки з datetime

In [ ]:
df3 = pd.read_csv(BASE + 'index_1.csv')
df3['datetime'] = pd.to_datetime(df3['datetime'])

df3['year'] = df3['datetime'].dt.year
df3['month'] = df3['datetime'].dt.month
df3['day'] = df3['datetime'].dt.day
df3['day_of_week'] = df3['datetime'].dt.dayofweek
df3['hour'] = df3['datetime'].dt.hour

df3[['datetime', 'year', 'month', 'day', 'day_of_week', 'hour']].head(10)

### Завдання 4. Обробка рядкових даних

In [ ]:
df4 = pd.DataFrame({
    'city': ['  Kyiv', 'LVIV ', ' Odesa ', 'kharkiv', 'DNIPRO  '],
    'phone': ['+38 (044) 111-22-33', '+38 (032) 222-33-44', '+38 (048) 333-44-55',
              '+38 (057) 444-55-66', '+38 (056) 555-66-77']
})
print("До очищення:")
print(df4)

df4['city'] = df4['city'].str.strip().str.lower()
df4['phone'] = df4['phone'].str.replace(r'[\s()+-]', '', regex=True)

print("\nПісля очищення:")
print(df4)

### Завдання 5. One-hot encoding для cash_type

In [ ]:
df5 = pd.get_dummies(df3, columns=['cash_type'])
df5.head()

**Інтерпретація:** замість колонки `cash_type` з'явилися дві бінарні колонки `cash_type_card` і `cash_type_cash`.
Значення `True` (1) у колонці означає, що транзакція належить до цієї категорії: у рядку з `cash_type_card = True` оплата була карткою, з `cash_type_cash = True` — готівкою.
У кожному рядку рівно одна з колонок має значення `True`.

### Завдання 6. Нормалізація та стандартизація money

In [ ]:
d6 = df3[['money']].copy()
d6['money_norm'] = (d6['money'] - d6['money'].min()) / (d6['money'].max() - d6['money'].min())
d6['money_std'] = (d6['money'] - d6['money'].mean()) / d6['money'].std()

print("money_norm: min =", d6['money_norm'].min(), ", max =", d6['money_norm'].max())
print("money_std:  mean =", d6['money_std'].mean(), ", std =", d6['money_std'].std())
d6.head()

### Завдання 7. Ознака is_card через np.where()

In [ ]:
df3['is_card'] = np.where(df3['cash_type'] == 'card', 1, 0)
df3[['cash_type', 'is_card']].head(10)

### ★ Завдання 8. Функція clean_dataset(df)

In [ ]:
def clean_dataset(df, date_col=None):
    """Повертає очищену копію DataFrame:
    1) видаляє повні дублікати;
    2) перетворює колонку(и) з датою на datetime64 (рядки з нерозпізнаною датою видаляються);
    3) заповнює числові пропуски медіаною, категоріальні — модою;
    4) видаляє рядки з викидами за методом IQR у будь-якій числовій колонці.
    Якщо date_col не вказано, датами вважаються колонки, у назві яких є 'date'."""
    out = df.copy().drop_duplicates()

    # дати
    if date_col is None:
        date_cols = [c for c in out.columns if 'date' in str(c).lower()]
    else:
        date_cols = [date_col]
    for c in date_cols:
        out[c] = pd.to_datetime(out[c], errors='coerce')
    if date_cols:
        out = out.dropna(subset=date_cols)

    # пропуски
    num_cols = out.select_dtypes(include='number').columns
    cat_cols = [c for c in out.columns if c not in num_cols and c not in date_cols]
    for c in num_cols:
        out[c] = out[c].fillna(out[c].median())
    for c in cat_cols:
        if out[c].isnull().any():
            out[c] = out[c].fillna(out[c].mode()[0])

    # викиди (IQR по всіх числових колонках)
    mask = pd.Series(True, index=out.index)
    for c in num_cols:
        q1, q3 = out[c].quantile(0.25), out[c].quantile(0.75)
        iqr = q3 - q1
        mask &= out[c].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    return out[mask].reset_index(drop=True)

In [ ]:
# Штучний «брудний» датасет
dirty = pd.DataFrame({
    'order_date': ['2024-05-01', '2024-05-02', '2024-05-03', '2024-05-04', '2024-05-05',
                   '2024-05-06', '2024-05-07', '2024-05-08', '2024-05-09', '2024-05-10', 'not a date'],
    'product':    ['Latte', 'Cocoa', None, 'Latte', 'Espresso', 'Latte', 'Cocoa', 'Espresso', 'Latte', 'Cocoa', 'Latte'],
    'price':      [38.7, 35.0, 36.0, None, 30.0, 38.7, 35.0, 30.0, 37.0, 5000.0, 38.0],
    'quantity':   [2, 1, 3, 2, 1, None, 2, 1, 3, 2, 1]
})
dirty = pd.concat([dirty, dirty.iloc[[0, 1]]], ignore_index=True)   # 2 дублікати

print("До очищення:", dirty.shape)
print(dirty.isnull().sum(), "\nДублікатів:", dirty.duplicated().sum())

cleaned = clean_dataset(dirty)

print("\nПісля очищення:", cleaned.shape)
print(cleaned.dtypes)
cleaned

In [ ]:
cleaned.to_csv(BASE + 'cleaned_demo.csv', index=False)
print("Результат збережено у cleaned_demo.csv")

# Відповіді на питання для самоперевірки
1. Пропуски, дублікати, викиди, неправильні типи даних, зайві пробіли та різний регістр у тексті. Їх усувають, бо вони спотворюють статистики (середнє, std), ламають сортування, групування й об'єднання та дають хибні висновки.
2. `df.isnull().sum()`.
3. Числові: `fillna(df['col'].mean())` або `median()`; категоріальні: `fillna(df['col'].mode()[0])`; або `dropna(subset=['col'])`.
4. `df.duplicated().sum()` — кількість; `df.drop_duplicates()` залишає перше входження (`keep='first'` за замовчуванням).
5. Викиди — значення, що сильно відхиляються від решти. IQR = Q3 − Q1; межі: `Q1 − 1.5·IQR` та `Q3 + 1.5·IQR`.
6. CSV не зберігає типи, тому дати читаються як текст (`object`). Перетворення: `pd.to_datetime(df['col'], errors='coerce')`.
7. `.str.strip()`, `.str.lower()`, `.str.upper()`, `.str.replace(...)`, `.str.contains(...)` тощо.
8. Створення нових колонок на основі наявних. Через `.dt` можна отримати рік, місяць, день, день тижня, годину; вони дозволяють аналізувати продажі за часом.
9. Кожна категорія стає окремою бінарною колонкою; `True`/1 означає належність рядка до категорії.
10. Нормалізація приводить значення до [0, 1] (min = 0, max = 1); стандартизація — до середнього 0 і std 1.
11. `df.to_csv('file.csv', index=False)`.
12. Щоб будь-хто міг повторити очищення і перевірити результат, а також щоб помилку було легко знайти й виправити.